# Exercice 5 : les cas et l'humain dans la boucle

**Mission :** donner une **mémoire** au SOAR (les cas) et rendre la parole à l'analyste pour les actions à risque (les approbations).

Deux défauts restent depuis l'exercice 2 :
1. **11 alertes, 11 rapports**, sans lien entre eux. À 09:20, le SOAR ne sait pas qu'il a déjà arrêté le mineur à 09:05 : il classe l'alerte en *faux positif*.
2. Désactiver un compte est une action **semi-automatique** (cours, chapitre 7) : un humain doit accepter. Jusqu'ici, le moteur acceptait tout seul.

**Ouvrez `minisoar/cas.py`**, puis comparez `playbooks/activite-hote.yaml` et `playbooks/activite-hote-cas.yaml` : une étape `cas` au début, et un verdict qui s'en sert.

In [1]:
import os
from fastapi.testclient import TestClient
from banc import scenario, environnement as env, rejeu
from minisoar import api, file, cas

os.environ["SOAR_SECRET_WEBHOOK"] = "cle-du-siem-8inf857"
os.environ["SOAR_SECRET_ANALYSTES"] = "cle-des-analystes-8inf857"     # une AUTRE clé
SIEM, ANALYSTE = {"x-soar-cle": "cle-du-siem-8inf857"}, {"x-soar-cle": "cle-des-analystes-8inf857"}

api.LANCER = api.lancer_en_file          # exercice 4 : la file
file.HUMAIN = cas                        # exercice 5 : approbations et cas
api.ROUTAGE.update({"R2-succes-apres-echecs": "connexion-suspecte-cas", "R3-compte-leurre": "connexion-suspecte-cas",
                    "R4-sortant-suspect": "activite-hote-cas", "R5-cpu": "activite-hote-cas"})
soar = TestClient(api.app)

def recevoir(alerte):
    """Le SIEM envoie une alerte ; un travailleur vide la file."""
    source, brut = scenario.a_envoyer(alerte)
    r = soar.post(f"/webhooks/{source}", json=brut, headers=SIEM).json()
    file.travailler(une_fois=True, travailleur="travailleur-1")
    return r

## 1. Un cas s'ouvre, un autre le rejoint

**Ce qu'on fait :** à 08:21, l'attaquant touche le compte leurre `oracle` (A04). Puis l'administratrice se trompe de mot de passe à 08:02 (A03, déjà passée, on l'envoie aussi).

In [2]:
alertes = scenario.construire(); api.VUES.clear()
for a in (alertes[2], alertes[3]):
    env.a_l_heure(a)
    recevoir(a)
cas.tableau()

,id,statut,severite,ouvert,alertes,titre
0,C001,fermé,moyenne,08:02:41,1,Connexion SSH réussie pour jtremblay après 2 é...
1,C002,en cours,critique,08:21:07,1,Tentative de connexion sur le compte leurre « ...


👀 **Ce qu'on doit voir :** deux cas. Celui de l'administratrice est **fermé** (faux positif) ; celui du leurre est **en cours**, sévérité critique. Lisez le cas du leurre :

In [3]:
print(soar.get("/cas/C002", headers=ANALYSTE).json()["rapport"])

# Cas C002 — Tentative de connexion sur le compte leurre « oracle » depuis 203.0.113.77
Statut : en cours · sévérité : critique · ouvert à 08:21:07

## Chronologie
- 08:21:07 [ouverture] cas ouvert par A04
- 08:21:07 [alerte] A04 R3-compte-leurre : Tentative de connexion sur le compte leurre « oracle » depuis 203.0.113.77
- 08:21:07 [note] [incident] 203.0.113.77 (interne : False) a fait 17 échec(s) sur admin, oracle, root, webadmin, premier essai à Dec 10 07:05:12. Comptes où elle est entrée : . Blocage : réussie. Désactivation : ignorée.

## Observables
- ip : 203.0.113.77
- compte : oracle, root, webadmin

## Mesures
- Début de l'attaque : 07:05:12 · détection : 08:21:07 · premier confinement : 08:21:07
- MTTD : 1:15:55 · MTTC : 0:00:00


❓ **Question pour la classe :** les **observables** du cas contiennent `root` (un des comptes essayés par l'attaquant). Pourquoi est-ce un problème pour le rattachement des alertes suivantes ?

*Réponse : **toute** future alerte qui mentionne `root` (et les robots essaient tous `root`) rejoindrait ce cas. Un observable trop commun relie des choses sans rapport. Les vrais SOAR ont des **listes d'exclusion** (comptes génériques, IP internes du SOC) et ne rattachent que sur des observables discriminants.*

## 2. L'intrusion : le SOAR demande la permission

**Ce qu'on fait :** à 08:41, l'attaquant entre sur `webadmin` (A06). Le playbook veut désactiver le compte, une étape marquée `approbation: true`.

In [4]:
env.a_l_heure(alertes[5])
r = recevoir(alertes[5])
print(file.etat(r["execution"])["statut"])
soar.get("/approbations", headers=ANALYSTE).json()

en attente


[{'execution': 'dabdcd27',
  'ref': 'desactiver',
  'action': 'comptes.desactiver',
  'args': '{"compte": "webadmin"}',
  'demande': '08:41:12'}]

👀 **Ce qu'on doit voir :** l'exécution est **en attente**. Le blocage de l'IP (automatique) est déjà fait ; la désactivation attend un humain.

❓ **Question pour la classe :** le SIEM a sa clé, les analystes ont la leur. Pourquoi deux clés ?

*Réponse : si la clé du SIEM suffisait pour approuver, quiconque la vole (elle est configurée dans un autre outil, parfois en clair) pourrait **approuver** des actions, ou envoyer une fausse alerte puis approuver sa propre réponse. Séparer les rôles : qui **signale** ne doit pas **décider**.*

**Ce qu'on fait :** l'analyste de garde répond **12 minutes plus tard** (il était en réunion), puis le travailleur reprend l'exécution.

In [5]:
demande = soar.get("/approbations", headers=ANALYSTE).json()[0]
print(soar.post(f"/approbations/{demande['execution']}/{demande['ref']}", headers=ANALYSTE,
                json={"decision": "approuver", "analyste": "jtremblay", "apres_minutes": 12}).json())
file.travailler(une_fois=True, travailleur="travailleur-1")
print(file.etat(r["execution"])["statut"], "—", env.comptes()["webadmin"])

{'execution': 'dabdcd27', 'ref': 'desactiver', 'decision': 'approuver', 'statut': 'remise en file'}
terminée — {'actif': False, 'groupes': ['www-data'], 'note': 'compte de service du site web'}


## 3. Rejouer toute la matinée

**Ce qu'on fait :** le rejeu complet. Un **analyste simulé** approuve chaque demande, toujours 12 minutes après. À la fin, le rapport final est celui du **cas principal**.

In [6]:
api.VUES.clear()

def envoyer(alerte):
    r = recevoir(alerte)
    for d in soar.get("/approbations", headers=ANALYSTE).json():         # l'analyste simulé
        soar.post(f"/approbations/{d['execution']}/{d['ref']}", headers=ANALYSTE,
                  json={"decision": "approuver", "analyste": "jtremblay", "apres_minutes": 12})
        file.travailler(une_fois=True, travailleur="travailleur-1")
    sortie = file.etat(r["execution"])["sortie"] if "execution" in r else {}
    return f"{sortie.get('rapport', r)}\nVERDICT : {sortie.get('verdict', '?')}"

def rapport_final():
    t = cas.tableau()
    return cas.rapport(t.sort_values("alertes").iloc[-1]["id"])

score = rejeu.rejouer(envoyer, "5 - cas et approbations", synthese=rapport_final)
cas.tableau()

07:28:03 A01 R1-force-brute           → attaque sans impact  ✅  (0 s)
07:34:10 A02 R1-force-brute           → attaque sans impact  ✅  (0 s)


08:02:41 A03 R2-succes-apres-echecs   → faux positif         ✅  (0 s)


08:21:07 A04 R3-compte-leurre         → incident             ✅  (0 s)
08:25:15 A05 R1-force-brute           → attaque sans impact  ✅  (0 s)


08:41:12 A06 R2-succes-apres-echecs   → incident             ✅  (0 s)


09:05:00 A07 R4-sortant-suspect       → incident             ✅  (0 s)
09:09:42 A08 R1-force-brute           → attaque sans impact  ✅  (0 s)
09:11:34 A09 R1-force-brute           → attaque sans impact  ✅  (0 s)


09:13:10 A10 R1-force-brute           → attaque sans impact  ✅  (0 s)
09:20:00 A11 R5-cpu                   → incident             ✅  (0 s)

Score : 83/100   (verdicts 35/35 · faits 30/35 · réponse 18/30 · pénalités −0)
Faits trouvés : ✅ début de l'attaque à 07:05 · ✅ script x.sh téléchargé · ✅ clé SSH de l'attaquant ajoutée · ✅ crontab de l'attaquant ajoutée · ✅ binaire /tmp/.x/kworkerds · ✅ reconnaissance (uname, whoami, /etc/passwd) · ❌ source de l'injection (198.51.100.23) signalée
Actions : ✅ IP de l'attaquant bloquée · ✅ compte webadmin désactivé · ❌ clé SSH de l'attaquant retirée · ❌ crontab de l'attaquant retirée · ✅ mineur arrêté
Rapport complet : resultats/rapport - 5 - cas et approbations.md   (0.0 min)


,id,statut,severite,ouvert,alertes,titre
0,C001,fermé,moyenne,08:02:41,1,Connexion SSH réussie pour jtremblay après 2 é...
1,C002,en cours,critique,08:21:07,4,Tentative de connexion sur le compte leurre « ...


👀 **Ce qu'on doit voir :** **11 verdicts sur 11** (l'alerte de 09:20 rejoint le cas de l'attaquant), et **un** cas principal avec 4 alertes. Ouvrez `resultats/rapport - 5 - cas et approbations.md` : la chronologie, la **fusion** de 09:20, les observables, les mesures.

In [7]:
cas.mesures(cas.tableau().sort_values("alertes").iloc[-1]["id"])

{'debut': '07:05:12',
 'detection': '08:21:07',
 'confinement': '08:21:07',
 'MTTD': '1:15:55',
 'MTTC': '0:00:00'}

❓ **Question pour la classe :** comment l'alerte de 09:05 (connexion sortante, qui ne parle ni de 203.0.113.77 ni de `webadmin`) a-t-elle rejoint le cas de l'attaquant ?

*Réponse : elle a d'abord ouvert **son propre** cas. Le playbook y a trouvé le processus `kworkerds` du compte `webadmin` ; `webadmin` est devenu un observable de ce cas. À 09:20, l'alerte CPU cite `webadmin` : elle touche les **deux** cas, qui **fusionnent**. C'est la mémoire qui relie des alertes que rien ne relie en apparence.*

❓ **Question pour la classe :** le MTTC est de 0 seconde (l'IP est bloquée dès l'alerte du leurre, à 08:21). Pourtant, le journal montre l'attaquant **entrer** à 08:41. Contradiction ?

*Réponse : le rejeu est un **enregistrement** : bloquer l'IP à 08:21 n'efface pas les lignes de 08:41. Dans la vraie vie, l'attaquant n'aurait **jamais** pu entrer. C'est la leçon du leurre : branché sur un playbook, il arrête l'attaque 20 minutes avant l'intrusion.*

❓ **Question pour la classe :** à 09:05, le SOAR a redemandé l'approbation de désactiver `webadmin`, déjà désactivé à 08:53. Quel est le coût de ces demandes inutiles ?

*Réponse : la **fatigue d'approbation** : un analyste qui reçoit 50 demandes par jour finit par tout accepter sans lire. Une demande ne devrait partir que si l'action change quelque chose (un `run_if` qui vérifie que le compte est encore actif).*

## 4. À vous : des cas pour les robots

Les robots de force brute ne créent pas de cas : leurs alertes disparaissent dans les rapports. Modifiez **`playbooks/force-brute.yaml`** (votre version de l'exercice 2) :
1. ajoutez au début une étape `cas` (action `cas.rattacher`, argument `alerte: ${{ TRIGGER }}`) ;
2. ajoutez `cas: ${{ ACTIONS.cas.result.cas }}` dans `returns`.

Le travailleur fermera tout seul les cas *attaque sans impact*. Relancez le rejeu (section 3), puis le test.

In [ ]:
t = cas.tableau()
robots = t[t.titre.str.startswith("5 échecs SSH")]
assert len(robots) >= 6, f"seulement {len(robots)} cas de robots"
assert (robots.statut == "fermé").all(), robots
print(f"✅ {len(robots)} cas de robots, tous fermés automatiquement.")
t

*Réponse : `solutions/playbooks/force-brute-cas.yaml`. Un SOC qui ferme 6 cas sur 8 automatiquement, c'est l'objectif réel d'un SOAR : que les analystes ne regardent que les 2 qui comptent.*

## Ce qu'il faut retenir
- Un **cas** regroupe les alertes d'un même incident, par leurs **observables** ; deux cas liés **fusionnent**. C'est la mémoire du SOAR.
- Choisir les observables est délicat : un observable trop commun (`root`) relie tout et n'importe quoi.
- Une étape à risque **met l'exécution en attente** ; elle reprend quand un analyste décide, avec **sa** clé (séparer qui signale et qui décide).
- Les cas donnent les **mesures** du chapitre (MTTD, MTTC), et le temps d'approbation y apparaît.
- Prochain exercice : l'**IA comme une étape** du playbook, pour ce qui demande du jugement.